# NB00 — Environment Setup on Google Colab

Run **once** at the beginning of each Colab session, **before** NB02. It:
1. Detects the GPU (expects H100 / A100 / RTX PRO 6000 Blackwell).
2. Mounts Google Drive.
3. **Copies the images `.zip` to Colab's local disk and unzips it there** — reading
   thousands of `.jpg`s directly from mounted Drive is VERY slow (each file = 1 network call);
   from local disk (`/content`) it's orders of magnitude faster.
4. Installs missing dependencies (timm).
5. Validates that the images referenced in the splits exist.

## Before running — what to upload to Drive
- The project folder: `A4-Triagem-Binaria/` (code + splits).
- A **single file** `Imgs.zip` with all `.jpg` images (zipping is faster for upload
  and for extracting than uploading the loose folder).

> **Zip tip (on your PC, inside `Data/`):** compress the `Imgs/` folder into an `Imgs.zip` file.
> On Windows: right-click the `Imgs` folder → *Send to* → *Compressed (zipped) folder*.


## 0. Parameters — adjust YOUR Drive paths

In [ ]:
# Project folder path ON DRIVE (where you uploaded A4-Triagem-Binaria):
PROJECT_ON_DRIVE = '/content/drive/MyDrive/A4-Triagem-Binaria'

# Image .zip path ON DRIVE:
IMAGES_ZIP_ON_DRIVE = '/content/drive/MyDrive/A4-Triagem-Binaria/Data/Imgs.zip'

# Where to unzip locally on Colab (fast, ephemeral disk):
LOCAL_IMAGES_DIR = '/content/Imgs'


## 1. Available GPU

In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi unavailable')
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB | Compute {p.major}.{p.minor}')


## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.isdir(PROJECT_ON_DRIVE), f'Project not found: {PROJECT_ON_DRIVE}'
assert os.path.isfile(IMAGES_ZIP_ON_DRIVE), f'Zip not found: {IMAGES_ZIP_ON_DRIVE}'
print('OK: project and zip found on Drive.')


## 3. Copy zip to local disk and extract
Copying the single zip to `/content` and extracting there is much faster than reading from mounted Drive.
The progress bar shows the extraction.


In [4]:
import shutil, zipfile, os, time
from tqdm.auto import tqdm

os.makedirs(LOCAL_IMAGES_DIR, exist_ok=True)

# (a) copy zip to local disk (fast: 1 large file)
local_zip = '/content/Imgs.zip'
if not os.path.exists(local_zip):
    t0 = time.time()
    print('Copying zip to local disk...')
    shutil.copy(IMAGES_ZIP_ON_DRIVE, local_zip)
    print(f'  copied in {time.time()-t0:.1f}s ({os.path.getsize(local_zip)/1e6:.0f} MB)')
else:
    print('zip is already on the local disk.')

# (b) extract with progress bar
with zipfile.ZipFile(local_zip) as z:
    members = z.namelist()
    # if the zip contains the 'Imgs/...' folder, extract in /content root; otherwise, in LOCAL_IMAGES_DIR
    has_top = members[0].split('/')[0]
    dest = '/content' if has_top.lower().startswith('imgs') else LOCAL_IMAGES_DIR
    print(f'Extracting {len(members)} items in {dest} ...')
    for m in tqdm(members, desc='unzip'):
        z.extract(m, dest)
print('Extraction completed.')


## 4. Locate the actual images folder (where the .jpgs are)

In [ ]:
import os
# search for the subfolder that actually contains .jpgs
def find_jpg_dir(roots):
    for root in roots:
        for dirpath, _, files in os.walk(root):
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
                return dirpath
    return None

IMG_DIR = find_jpg_dir(['/content/Imgs', LOCAL_IMAGES_DIR, '/content'])
assert IMG_DIR, 'Could not find any .jpg image after extraction!'
n_imgs = sum(1 for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg','.jpeg','.png')))
print(f'Images found in: {IMG_DIR}')
print(f'Total image files: {n_imgs}')


## 5. Install missing dependencies (timm)

In [6]:
import importlib, subprocess, sys
def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg); print(f'  {pkg}: already installed')
    except ImportError:
        print(f'  installing {pip_name or pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg], check=True)
for pkg, pip in [('timm','timm>=0.9.12'), ('sklearn','scikit-learn'), ('tqdm','tqdm')]:
    ensure(pkg, pip)
print('deps OK.')


## 6. Export environment variables to config
`config.py` reads `A4_IMAGES_DIR`. We define it here pointing to the fast local disk.
**Important:** this also needs to be valid in NB02. Since NB02 runs in the same runtime,
we save a file that the config loads, and also set the env for this session.


In [7]:
import os
os.environ['A4_IMAGES_DIR'] = IMG_DIR

# writes a simple .env that the config can optionally read (robustness between notebooks)
with open(os.path.join(PROJECT_ON_DRIVE, '.a4_env'), 'w') as f:
    f.write(f'A4_IMAGES_DIR={IMG_DIR}\n')

print('A4_IMAGES_DIR =', os.environ['A4_IMAGES_DIR'])
print('In NB02, the 1st cell must contain BEFORE importing config:')
print(f"    os.environ['A4_IMAGES_DIR'] = '{IMG_DIR}'")


## 7. Final validation — do splits match the images?

In [8]:
import os, sys, glob
import pandas as pd

# finds the splits inside the project on Drive
splits_dir = os.path.join(PROJECT_ON_DRIVE, 'splits')
assert os.path.isdir(splits_dir), f'splits not found in {splits_dir}'

# load all referenced image_name
csvs = glob.glob(os.path.join(splits_dir, 'fold_*_*.csv'))
referenced = set()
for c in csvs:
    df = pd.read_csv(c)
    referenced |= set(df['image_name'].astype(str))

present = set(os.listdir(IMG_DIR))
missing = [r for r in referenced if r not in present]
print(f'images referenced in splits: {len(referenced)}')
print(f'present on disk: {len(present & referenced)}')
print(f'MISSING: {len(missing)}')
if missing:
    print('  missing examples:', missing[:5])
    print('  >> Check if the zip contains ALL images.')
else:
    print('\n✅ ALL SET. Open NB02, adjust the 1st cell with the A4_IMAGES_DIR above,')
    print('   set QUICK_TEST=True for the smoke test and then QUICK_TEST=False for full training.')
